In [1]:
# Utility function to ensure DataFrames with geometry are converted to GeoDataFrames
import geopandas as gpd
import pandas as pd
from shapely import wkt

def ensure_geodataframe(df, geometry_col='geometry'):
    """
    Ensures that a DataFrame with geometry column is converted to a GeoDataFrame.
    If conversion fails, tries to robustly decode geometry values before failing.
    
    Args:
        df: DataFrame or GeoDataFrame
        geometry_col: Name of the geometry column
    
    Returns:
        GeoDataFrame with proper CRS set
    """
    import shapely
    import binascii

    def try_decode_geometry(val):
        """
        Try to decode a geometry value that may be:
        - Already a shapely geometry
        - A WKT string
        - A WKB hex string (bytes or str)
        - A WKB bytes object
        If it cannot be decoded, returns None.
        """
        if isinstance(val, shapely.geometry.base.BaseGeometry):
            return val
        if val is None or (isinstance(val, float) and pd.isna(val)):
            return None
        # Try WKT
        if isinstance(val, str):
            try:
                # Try WKT first
                return shapely.wkt.loads(val)
            except Exception:
                pass
            try:
                # Try WKB hex string
                return shapely.wkb.loads(binascii.unhexlify(val))
            except Exception:
                pass
        # Try WKB bytes
        if isinstance(val, (bytes, bytearray)):
            try:
                return shapely.wkb.loads(val)
            except Exception:
                pass
        return None

    # If already a GeoDataFrame, ensure CRS is set
    if isinstance(df, gpd.GeoDataFrame):
        if df.crs is None:
            df = df.set_crs(epsg=4326)  # WGS84
        return df

    # If regular DataFrame with geometry column, convert to GeoDataFrame
    if geometry_col in df.columns:
        # Convert geometry column from WKT strings to geometry objects if needed
        if df[geometry_col].dtype == 'object':
            try:
                df[geometry_col] = df[geometry_col].apply(wkt.loads)
            except Exception:
                # If wkt.loads fails, try to robustly decode geometry values
                try:
                    df[geometry_col] = df[geometry_col].apply(try_decode_geometry)
                    # Remove rows where geometry could not be decoded
                    n_invalid = df[geometry_col].isna().sum()
                    if n_invalid > 0:
                        print(f"⚠️ {n_invalid} rows had invalid geometry and will be dropped.")
                        df = df[df[geometry_col].notna()]
                except Exception as e:
                    print(f"❌ Could not decode geometry: {e}")
                    raise

        # Convert to GeoDataFrame
        try:
            df = gpd.GeoDataFrame(df, geometry=geometry_col)
        except Exception as e:
            # Try to robustly decode geometry and try again
            try:
                df[geometry_col] = df[geometry_col].apply(try_decode_geometry)
                n_invalid = df[geometry_col].isna().sum()
                if n_invalid > 0:
                    print(f"⚠️ {n_invalid} rows had invalid geometry and will be dropped.")
                    df = df[df[geometry_col].notna()]
                df = gpd.GeoDataFrame(df, geometry=geometry_col)
            except Exception as e2:
                print(f"❌ Could not convert to GeoDataFrame after robust decode: {e2}")
                raise

        # Set CRS if not already set
        if df.crs is None:
            df = df.set_crs(epsg=4326)  # WGS84

        return df

    # Return as-is if no geometry column found
    return df

print("✅ Utility function loaded: ensure_geodataframe()")


✅ Utility function loaded: ensure_geodataframe()


In [10]:
import sys
import pandas as pd
import geopandas as gpd
import os
from datetime import datetime

sys.path.append('..')  # Add parent directory to path

# Import from local utility modules
from lvt_utils import ensure_geodataframe
from cloud_utils import get_feature_data_with_geometry

# Set pandas to display all columns
pd.set_option('display.max_columns', None)

# Define Baltimore Realproperty_OB layer endpoint components
dataset_name = "12/query"
base_url = "https://maps.co.ramsey.mn.us/arcgis/rest/services/OpenData/OpenData/FeatureServer"
layer_id = 12

# Set scrape variable as needed
data_scrape = 1  # set to 0 or 1 as required

save_dir = os.path.join("data", "baltimore")
os.makedirs(save_dir, exist_ok=True)

if data_scrape == 1:
    # Download data with geometry (paginate=True to pull all records)
    baltimore_gdf = get_feature_data_with_geometry(dataset_name, base_url, layer_id, paginate=True)
    # Make sure it's a GeoDataFrame
    baltimore_gdf = ensure_geodataframe(baltimore_gdf)
    # Compose filename with date
    today_str = datetime.now().strftime("%Y%m%d")
    fname = f"baltimore_{today_str}.gpq"
    fpath = os.path.join(save_dir, fname)
    baltimore_gdf.to_parquet(fpath)
    display(baltimore_gdf.head())
else:
    # Find most recent geoparquet in data/baltimore
    try:
        files = [f for f in os.listdir(save_dir) if f.lower().endswith(".gpq")]
        if not files:
            raise FileNotFoundError("No baltimore geoparquet scrapes found in data/baltimore/")

        # Get the latest by date in filename
        files_with_dates = []
        for fname in files:
            try:
                date_str = fname.split("_")[1].split(".")[0]
                dt = datetime.strptime(date_str, "%Y%m%d")
                files_with_dates.append((dt, fname))
            except Exception:
                continue
        if not files_with_dates:
            raise FileNotFoundError("No valid baltimore geoparquet scrapes found in data/baltimore/")
        latest_fname = max(files_with_dates, key=lambda x: x[0])[1]
        fpath = os.path.join(save_dir, latest_fname)
        baltimore_gdf = gpd.read_parquet(fpath)
        display(baltimore_gdf.head())
    except Exception as e:
        raise RuntimeError(f"Failed to find any previous scraped data: {e}")


Layer metadata CRS WKID: None
⚠️ Could not detect layer CRS from metadata. Defaulting to EPSG:3857 (risky).
Total records in 12/query: 167662
Query response spatialReference: {'wkid': 4326, 'latestWkid': 4326}
Fetched records 0 to 1000
Fetched records 1000 to 2000
Fetched records 2000 to 3000
Fetched records 3000 to 4000
Fetched records 4000 to 5000
Fetched records 5000 to 6000
Fetched records 6000 to 7000
Fetched records 7000 to 8000
Fetched records 8000 to 9000
Fetched records 9000 to 10000
Fetched records 10000 to 11000
Fetched records 11000 to 12000
Fetched records 12000 to 13000
Fetched records 13000 to 14000
Fetched records 14000 to 15000
Fetched records 15000 to 16000
Fetched records 16000 to 17000
Fetched records 17000 to 18000
Fetched records 18000 to 19000
Fetched records 19000 to 20000
Fetched records 20000 to 21000
Fetched records 21000 to 22000
Fetched records 22000 to 23000
Fetched records 23000 to 24000
Fetched records 24000 to 25000
Fetched records 25000 to 26000
Fetche

,OBJECTID,CountyID,ParcelID,FIPsCodeParcelID,RollType,BuildingNumber,BuildingNumberSuffix,UnitType,UnitNumber,Unit,StreetPrefixDirection,StreetPrefixType,StreetName,StreetSuffixType,StreetSuffixDirection,StreetNameAll,SiteAddress,SiteCityNameUSPS,SiteCityNameCode,SiteCityName,SiteZIP5,SiteZIP4,SiteZIP,SiteCityStateZIP,OwnershipCategory,OwnerLastName,OwnerName,OwnerName1,OwnerName2,OwnerAddress1,OwnerAddress2,OwnerCityStateZIP,TaxName1,TaxName2,TaxAddress1,TaxAddress2,TaxCityStateZIP,HomesteadName1,HomesteadName2,HomesteadAddress1,HomesteadAddress2,HomesteadCityStateZIP,NeighborhoodCode,MunicipalCode,TIFDistrict,SchoolDistrictNumber,SchoolDistrictName,WatershedIDTax,WatershedDistrictNameTax,WatershedDistNamePoly,PlatID,PlatName,TaxDescription,Block,Lot,ParcelAcresDeed,ParcelSquareFeet,ParcelAcresPolygon,ParcelFrontage,TaxYear,EMVYear,EMVLand,EMVBuilding,EMVTotal,TotalTax,SpecialAssessmentDue,TaxCapacity,CostLandValue,TaxYear1,EMVYear1,EMVLand1,EMVBuilding1,EMVTotal1,TotalTax1,SpecialAssessmentDue1,TaxYear2,EMVYear2,EMVLand2,EMVBuilding2,EMVTotal2,TotalTax2,SpecialAssessmentDue2,LandmarkBusinessName,LandUseCode,LandUseCodeDescription,MultipleUseYN,UseTypeCode1,UseType1,UseTypeCode2,UseType2,UseTypeCode3,UseType3,UseTypeCode4,UseType4,TaxExemptYN,ExemptUse1,ExemptUse2,ExemptUse3,ExemptUse4,GreenAcresYN,OpenSpaceYN,AgriculturalPreserveYN,AgPreserveEnrolled,AgPreserveExpire,HomesteadYN,HomesteadDescription,StructureCode,StructureDescription,DwellingType,LivingUnit,HomeStyleCode,HomeStyleDescription,ExteriorWallCode,ExteriorWallDescription,Stories,RoomTotal,BedRoom,FamilyRoom,BasementYN,HeatSystemCode,HeatSystemType,HeatCode,HeatType,LivingAreaSquareFeet,BusinessSquareFeet,GarageYN,GarageSquareFeet,YearBuilt,EffectiveYearBuilt,TopologyCode,TopologyDescription,UtilityCode,UtilityDescription,LastSaleDate,SalePrice,InspectionYear,InspectionStatus,X,Y,Latitude,Longitude,Section,Township,Range,QuarterQuarter,PolygonPointRelationship,PropertyDataJoinDate,InspectionDataJoinDate,EditDate,geometry
0,30461582,27123,122922340241,27123-122922340241,RP,2480,None,None,None,None,None,None,7TH,AVE,E,7TH AVE E,2480 7TH AVE E,NORTH ST PAUL,02395261,NORTH SAINT PAUL,55109,2906,55109-2906,NORTH SAINT PAUL MN 55109-2906,Unknown,FLEX HOLDING LLC,FLEX HOLDING LLC,FLEX HOLDING LLC,None,2480 7TH AVE E,None,NORTH ST PAUL MN 55109-2906,FLEX HOLDING LLC,None,2480 7TH AVE E,None,NORTH ST PAUL MN 55109-2906,None,None,None,None,None,CS414000,69,None,0622,North St. Paul/Maplewood/Oakdale School District,034,METRO WATERSHED,Ramsey-Washington Metro WSD,03889,"FIRST ADDITION TO NORTH ST.,PA","FIRST ADDITION TO NORTH ST.,PA LOTS 9 THRU LO...",4,9,0.41,17859.6,0.40,NaN,2026.0,2025.0,144000.0,1934800.0,2078800.0,NaN,None,40826.0,144000.0,2025.0,2024.0,144000.0,2760600.0,2904600.0,96082.00,NaN,2024.0,2023.0,144000.0,2773100.0,2917100.0,93424.00,NaN,Refelx Medical Moding,340,MANUFACTURING AND ASSEMBLY LIGHT,N,234,3A INDUSTRIAL LAND AND BUILDING,None,None,None,None,None,None,N,None,None,None,None,N,N,N,None,None,N,None,405,RESEARCH & DEVELOPMENT,None,0.0,None,None,None,None,NaN,NaN,NaN,NaN,None,None,None,None,None,NaN,28650.0,N,NaN,2010.0,2019.0,1,LEVEL,1,ALL PUBLIC,1.255478e+12,190000.0,2023,Property reviewed 3/12/2025,600102.11,179893.42,45.009573,-92.996316,12,29,22,SESW,1,1.768522e+12,1.767312e+12,1395531174000,"POLYGON ((-92.99633 45.00936, -92.99641 45.009..."
1,30461583,27123,283022130010,27123-283022130010,RP,3860,None,None,None,None,None,None,LABORE,RD,None,LABORE RD,3860 LABORE RD,VADNAIS HEIGHTS,02397106,VADNAIS HEIGHTS,55110,4128,55110-4128,VADNAIS HEIGHTS MN 55110-4128,Unknown,SLP CENTER LLC,SLP CENTER LLC,SLP CENTER LLC,None,3860 LABORE RD,None,SAINT PAUL MN 55110-9787,SLP CENTER LLC,None,3860 LABORE RD,None,SAINT PAUL MN 55110-9787,None,None,None,None,None,C0214000,89,None,0624,White Bear Lake School District,None,None,Vadnais Lake Area WMO,04724,REGISTERED LAND SURVEY 69,REGISTERED LAND SURVEY 69 TRACT A,None,A,0.95,41382.0,0.97,NaN,2026.0,2025.0,165500.0,2

In [2]:
import sys
import pandas as pd
sys.path.append('..')  # Add parent directory to path
from cloud_utils import get_feature_data, get_feature_data_with_geometry
from lvt_utils import model_split_rate_tax, calculate_current_tax, model_full_building_abatement, model_stacking_improvement_exemption
from census_utils import get_census_data, get_census_blockgroups_shapefile, get_census_data_with_boundaries, match_to_census_blockgroups

scrape_data = 0

In [9]:

import os
from datetime import datetime
import glob

scrape_data = 1

# Directory to save/load data
data_dir = "data/st_paul"
os.makedirs(data_dir, exist_ok=True)

if scrape_data == 1:
    # Base URL for the ArcGIS services
    base_url = "https://maps.co.ramsey.mn.us/arcgis/rest/services/OpenData/OpenData/FeatureServer"
    # Fetch the main parcel dataset with tax info
    parcel_civic_df = get_feature_data_with_geometry('18/query', base_url)
    # Save with geometry to parquet, with current date
    today_str = datetime.now().strftime("%Y_%m_%d")
    out_path = os.path.join(data_dir, f"st_paul_parcels_{today_str}.parquet")
    parcel_civic_df.to_parquet(out_path, index=False)
    print(f"Saved new scrape to {out_path}")
else:
    # Find the most recent parquet file in the data_dir
    files = glob.glob(os.path.join(data_dir, "st_paul_parcels_*.parquet"))
    if not files:
        raise FileNotFoundError("No previously scraped parcel files found in data/st_paul/")
    # Sort files by date in filename
    files_sorted = sorted(files, key=lambda x: datetime.strptime(os.path.basename(x).replace("st_paul_parcels_", "").replace(".parquet", ""), "%Y_%m_%d"), reverse=True)
    latest_file = files_sorted[0]
    print(f"Loading most recent scrape: {latest_file}")
    parcel_civic_df = pd.read_parquet(latest_file)

# Ensure parcel_civic_df is a proper GeoDataFrame
parcel_civic_df = ensure_geodataframe(parcel_civic_df)
print(f"✅ Parcel data loaded as {type(parcel_civic_df).__name__} with CRS: {parcel_civic_df.crs}")


Layer metadata CRS WKID: None
⚠️ Could not detect layer CRS from metadata. Defaulting to EPSG:3857 (risky).
Query response spatialReference: {'wkid': 4326, 'latestWkid': 4326}
Fetched records 0 to 1000


AttributeError: 'NoneType' object has no attribute 'to_parquet'

In [6]:
print(parcel_civic_df.columns)

Index(['OBJECTID', 'ParcelID', 'TaxType', 'TaxSubType', 'Counter',
       'SourceReference', 'LegalStartDate', 'ModificationDate', 'Accuracy',
       'LyrName', 'TaxMapFeature', 'geometry'],
      dtype='object')


In [8]:
parcel_civic_df
print(parcel_civic_df["TaxType"].unique())

['Real Property']
